In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path(os.getcwd())
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

print("Setup done.")

Setup done.


In [2]:
import json
import pandas as pd
from pathlib import Path

from graph.pipeline import pipeline
from graph.state import AgentState
from storage.database import init_db
from storage.trace_store import save_trace, save_agent_log, save_eval_result
from eval.hallucination import calculate_hallucination_rate
from eval.cost_tracker import calculate_cost
from eval.latency import calculate_latency_breakdown
from eval.failure_taxonomy import classify_failure

init_db()
print("Importing modules — done.")

/Users/burakegekaya/Desktop/AgentBench-TR/venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Initializing database — creating tables: traces, agent_logs, eval_results.
Importing modules — done.


In [4]:
with open("data/eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)

print(f"Loading eval questions — {len(questions)} questions loaded.")

Loading eval questions — 35 questions loaded.


In [5]:
results = []

for q in questions:
    print(f"\nRunning query [{q['id']}]: {q['question'][:60]}")

    state = AgentState(query=q["question"])
    result = pipeline.invoke(state)

    trace_id = save_trace(
        query_text=q["question"],
        final_answer=result["final_answer"],
        confidence_score=result["confidence_score"],
    )

    for log in result["agent_logs"]:
        save_agent_log(
            trace_id=trace_id,
            agent_name=log["agent"],
            input=str(log["input"]),
            output=str(log["output"]),
            latency_ms=log["latency_ms"],
        )

    hallucination = calculate_hallucination_rate(trace_id)
    cost          = calculate_cost(trace_id)
    latency       = calculate_latency_breakdown(trace_id)
    failure       = classify_failure(AgentState(
        query=q["question"],
        retrieved_docs=result["retrieved_docs"],
        confidence_score=result["confidence_score"],
        retry_count=result["retry_count"],
        agent_logs=result["agent_logs"],
    ))

    results.append({
        "id":                 q["id"],
        "category":           q["category"],
        "question":           q["question"],
        "ground_truth":       q["ground_truth"],
        "final_answer":       result["final_answer"],
        "confidence_score":   result["confidence_score"],
        "retry_count":        result["retry_count"],
        "hallucination_rate": hallucination["hallucination_rate"],
        "cost_usd":           cost["cost_usd"],
        "total_latency_ms":   latency["total_ms"],
        "bottleneck_agent":   latency["bottleneck_agent"],
        "failure_type":       failure,
        "trace_id":           trace_id,
    })

    print(f"  confidence={result['confidence_score']} | failure={failure} | cost=${cost['cost_usd']}")

print(f"\nRunning all queries — {len(results)} completed.")


Running query [q001]: BERTurk ne zaman yayınlandı?
Running PlannerAgent — generated 4 sub-tasks in 2610ms.
Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 89 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 2 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from IntroToAINotlarım_cleaned.txt...
  Generated 138 chunks.
Loading chunks from LANGCHAIN_README_cleaned.txt...
  Generated 2 chunks.
Loading chunks from LANGGRAPH_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from Multimodal_LLM_Paper_cleaned.txt...
  Generated 6 chunks.
Loading chunks from THQuAD_cleaned.txt...
  Generated 7 chunks.
Loading complete — 249 total chunks.
Building BM25 index...
Building complete — 249 chunks indexed.
Initializing ChromaDB collection at /Users/burakegekaya/Desktop/AgentBench-TR/storage/chroma...
Collection 'agentbench

In [6]:
Path("results").mkdir(exist_ok=True)
df = pd.DataFrame(results)
df.to_csv("results/eval_results.csv", index=False, encoding="utf-8")
print(f"Saving eval results — {len(df)} rows written to results/eval_results.csv.")
print(df[["id","category","confidence_score","hallucination_rate","cost_usd","total_latency_ms","failure_type"]].to_string())

Saving eval results — 35 rows written to results/eval_results.csv.
      id    category  confidence_score  hallucination_rate  cost_usd  total_latency_ms      failure_type
0   q001     factual              1.00              0.0000  0.000033          15854.43              None
1   q002     factual              1.00              0.0000  0.000034           5096.47              None
2   q003     factual              1.00              0.0000  0.000047          14920.38              None
3   q004     factual              1.00              0.0000  0.000051           7246.94              None
4   q005     factual              1.00              0.0000  0.000057          21380.12              None
5   q006  conceptual              1.00              0.0000  0.000051          10049.57              None
6   q007  conceptual              1.00              0.0000  0.000048           4197.60              None
7   q008  conceptual              1.00              0.0000  0.000051           5660.79       

In [7]:
print("=== AVERAGES ===")
print(f"Consistency (confidence proxy) : {df['confidence_score'].mean():.4f}")
print(f"Hallucination Rate             : {df['hallucination_rate'].mean():.4f}")
print(f"Avg Cost / Query (1 run)       : ${df['cost_usd'].mean():.6f}")
print(f"Avg Cost / Query (10 runs est) : ${df['cost_usd'].mean() * 10:.6f}")
print(f"Total Cost (1 run x 35 q)      : ${df['cost_usd'].sum():.6f}")
print(f"Avg Latency / Query            : {df['total_latency_ms'].mean():.0f}ms")
print(f"Most common bottleneck         : {df['bottleneck_agent'].mode()[0]}")
print(f"Failure count                  : {df['failure_type'].notna().sum()} / {len(df)}")
print(f"Most common failure type       : {df['failure_type'].dropna().mode()[0] if df['failure_type'].notna().any() else 'None'}")

=== AVERAGES ===
Consistency (confidence proxy) : 0.9049
Hallucination Rate             : 0.0952
Avg Cost / Query (1 run)       : $0.000057
Avg Cost / Query (10 runs est) : $0.000567
Total Cost (1 run x 35 q)      : $0.001985
Avg Latency / Query            : 9380ms
Most common bottleneck         : validator
Failure count                  : 8 / 35
Most common failure type       : TOOL_CALL_FAILED


In [8]:
for row in results:
    save_eval_result(
        trace_id=row["trace_id"],
        consistency_score=row["confidence_score"],
        hallucination_rate=row["hallucination_rate"],
        cost_usd=row["cost_usd"],
    )
print("Saving eval results to SQLite — done.")

Saving eval result for trace 0c9929b5... — done.
Saving eval result for trace dce14f5e... — done.
Saving eval result for trace 09cf7a38... — done.
Saving eval result for trace c937c325... — done.
Saving eval result for trace 548deda2... — done.
Saving eval result for trace b5d011dd... — done.
Saving eval result for trace 8d3fd21b... — done.
Saving eval result for trace a88e88cd... — done.
Saving eval result for trace 6fdabc36... — done.
Saving eval result for trace b05cb2df... — done.
Saving eval result for trace 07c7f2b2... — done.
Saving eval result for trace f1099429... — done.
Saving eval result for trace db750900... — done.
Saving eval result for trace 01371288... — done.
Saving eval result for trace b9336a71... — done.
Saving eval result for trace e148e264... — done.
Saving eval result for trace fbfb7546... — done.
Saving eval result for trace 64892d51... — done.
Saving eval result for trace 9b6a4d2a... — done.
Saving eval result for trace f988e47f... — done.
Saving eval result f